# ResNet18 粗糙度分类训练（Colab GPU 版）

**使用方法：**
1. 修改右上角运行时 → T4 GPU
2. 上传 `data/EDM/50X/` 整个文件夹到 Colab
3. 依次运行每个 Cell
4. 训练完成后下载 `resnet18.pth` 和 `class_to_idx.json`

In [ ]:
# 1. 检查 GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 2. 安装依赖（如果还没装）
!pip install scikit-image -q

In [ ]:
# 3. 检查数据集（确保 data/EDM/50X/ 已上传）
import os
data_dir = 'data/EDM/50X'
if os.path.exists(data_dir):
    classes = sorted(os.listdir(data_dir))
    print(f'找到 {len(classes)} 个类: {classes}')
    for c in classes:
        n = len(os.listdir(os.path.join(data_dir, c)))
        print(f'  {c}: {n} 张')
else:
    print(f'错误: {data_dir} 不存在，请先上传数据集！')

In [ ]:
# 4. ResNet18 训练（完整脚本）
import json, torch, torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

DATA_DIR = 'data/EDM/50X'
EPOCHS, BATCH, LR = 15, 32, 1e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('使用设备:', device)

# 数据变换
train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
test_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# 数据划分（修复版：分别创建带不同 transform 的 dataset）
full = datasets.ImageFolder(DATA_DIR)
n_test = int(len(full) * 0.2)
n_total = len(full)
indices = torch.randperm(n_total, generator=torch.Generator().manual_seed(42)).tolist()
train_indices = indices[:n_total - n_test]
test_indices = indices[n_total - n_test:]

train_full = datasets.ImageFolder(DATA_DIR, transform=train_tf)
test_full = datasets.ImageFolder(DATA_DIR, transform=test_tf)
train_set = Subset(train_full, train_indices)
test_set = Subset(test_full, test_indices)

train_loader = DataLoader(train_set, BATCH, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, BATCH, num_workers=2)
num_classes = len(full.classes)
print(f'{len(full)} 张图，{num_classes} 类')

# 模型
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# 训练
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(model(imgs), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval(); correct = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            pred = model(imgs.to(device)).argmax(1)
            correct += (pred == labels.to(device)).sum().item()
    acc = correct / n_test
    print(f'epoch {epoch+1}/{EPOCHS}  loss={loss.item():.4f}  test_acc={acc:.4f}')
    if acc > best_acc:
        best_acc = acc
        Path('models').mkdir(exist_ok=True)
        torch.save(model.state_dict(), 'models/resnet18.pth')
        json.dump(full.class_to_idx, open('models/class_to_idx.json', 'w'))

print(f'\n最佳测试准确率: {best_acc:.4f}')
print('模型已保存到 models/resnet18.pth')

## 5. 下载训练好的模型

运行下面的 cell，会自动下载两个文件到本地：
- `resnet18.pth` — 模型权重
- `class_to_idx.json` — 类别映射

下载后放到本地项目的 `models/` 目录下。

In [ ]:
# 6. 下载模型到本地
from google.colab import files
print('正在下载 resnet18.pth ...')
files.download('models/resnet18.pth')
print('正在下载 class_to_idx.json ...')
files.download('models/class_to_idx.json')
print('下载完成！请放到本地项目的 models/ 目录下')